# SEC Document Service — Raw Disclosure Documents (market.sec.or.th)

List and **download the original documents** companies file with the Thai SEC's information-disclosure
system (IDISC): the financial-statement **Excel** package, **Form 56-1**, **Form 56-2**, Key Financial
Ratio, and MD&A — for any SET/mai-listed issuer.

**You will learn how to:**
- Resolve a symbol/name to its SEC company record
- List available documents across all five categories (with complete large sections)
- Filter by category and date window
- Download a document to bytes and save it to disk (and peek inside the zip)
- Download many concurrently, and handle dead links
- Use the `SecCompany` facade

> ⚠️ **Notes:** this is a **different host** from the SET services (`market.sec.or.th`, an ASP.NET site).
> Dates are **dd/mm/yyyy**. Some links are dead on the SEC side and return an HTML "file not found" page
> under HTTP 200 — the download service detects this and raises `FetchError`.

## 1. Setup

In [ ]:
import io
import tempfile
import zipfile
from collections import Counter

from settfex.services.sec import (
    DocumentCategory,
    SecCompany,
    download_sec_document,
    download_sec_documents,
    get_sec_documents,
    resolve_company,
)

# In Jupyter, await works directly (no asyncio.run needed)
SYMBOL = "CPALL"

## 2. Resolve a company

Map a symbol or name to the SEC `uniqueIDReference` (the join key).

In [ ]:
company = await resolve_company(SYMBOL)
print(company.company_name, "->", company.unique_id, "| primary:", company.is_primary)

## 3. List documents — all five categories

Without a date window the SEC form uses its own default; pass `from_date`/`to_date` for history.
Large sections are completed automatically (`follow_view_more=True`).

In [ ]:
docs = await get_sec_documents(SYMBOL, from_date="01/01/2024", to_date="31/12/2026")
print(f"total: {len(docs)}")
print(dict(Counter(d.category.value for d in docs)))

## 4. Filter by category

Pass `DocumentCategory` members or their string values.

In [ ]:
fs_and_561 = await get_sec_documents(
    SYMBOL,
    types=[DocumentCategory.FINANCIAL_STATEMENT, "form_56_1"],
    from_date="01/01/2024", to_date="31/12/2026",
)
for d in fs_and_561[:8]:
    print(f"{d.category.value:20} y={d.year} {d.statement_type or '':12} {d.period or '':4}"
          f" {d.status or '':9} {d.file_kind or '?'}")

## 5. Inspect a `SecDocument`

In [ ]:
doc = next(d for d in docs if d.category == DocumentCategory.FINANCIAL_STATEMENT and d.file_kind == "zip")
for field, value in doc.model_dump().items():
    print(f"{field:16}: {value}")

## 6. Download one document — and peek inside the zip

`download_sec_document` returns a `DownloadedFile` (filename + raw `bytes`). A financial-statement
package is a zip containing the original `FINANCIAL_STATEMENTS.XLSX` plus the auditor report and notes.

In [ ]:
file = await download_sec_document(doc)
print(f"{file.filename}  ({file.size:,} bytes, {file.content_type})")
print("contents:", zipfile.ZipFile(io.BytesIO(file.content)).namelist())

# Save to disk with .save(dir_or_path)
# file.save("./sec_downloads")

## 7. Download many concurrently, saving to a folder

In [ ]:
zips = [d for d in fs_and_561 if d.file_kind == "zip"][:3]
with tempfile.TemporaryDirectory() as tmp:
    got = await download_sec_documents(zips, dest_dir=tmp, max_concurrency=3)
    from pathlib import Path
    print(f"downloaded {len(got)} file(s); on disk: {[p.name for p in Path(tmp).glob('*')]}")

## 8. Dead links (soft 404)

Some links are dead on the SEC side and return an HTML "file not found" page under HTTP 200. The
download service detects this and raises `FetchError` (in a batch, such items are skipped by default).

In [ ]:
from settfex.exceptions import FetchError

f56_2 = await get_sec_documents(SYMBOL, types="form_56_2", from_date="01/01/2024", to_date="31/12/2026")
dead = next((d for d in f56_2 if "dat/annual" in (d.file_id or "")), None)
if dead:
    try:
        await download_sec_document(dead)
    except FetchError as exc:
        print("correctly raised:", str(exc)[:80])
else:
    print("no dead dat/annual link for this symbol right now")

## 9. The `SecCompany` facade

In [ ]:
sec = SecCompany("CPALL")
await sec.resolve()
recent = await sec.list_documents(types="form_56_1", from_date="01/01/2022", to_date="31/12/2026")
print(f"{len(recent)} Form 56-1 document(s)")
for d in recent:
    print(f"  {d.year}  received {d.receive_date}  {d.file_url}")

## Use Cases

- **Fundamental research / AI pipelines**: pull the original `FINANCIAL_STATEMENTS.XLSX` for a universe of
  symbols and parse it yourself (finer detail than the modeled SET factsheet).
- **Annual-report archives**: bulk-download Form 56-1 / 56-2 One Reports per issuer per year.
- **Disclosure monitoring**: list recent documents on a schedule and download new filings.

**See also**: `docs/settfex/services/sec/financial_report.md` · SET Financial Statements (notebook 11) for the
*modeled* balance sheet/income/cash-flow · News (notebook 16) for SET company news.